In [25]:
#import and setup
import json
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path

In [26]:
#Load disease classes
PROJECT_DIR = Path(
    r"D:\Pradeep Pandey\Nexvia Technology  Intership\Project 1"
)

with open(PROJECT_DIR / "processed" / "disease_classes.json", "r") as f:
    disease_classes = json.load(f)

num_classes = len(disease_classes)

print("Number of disease classes:", num_classes)
print(disease_classes)

Number of disease classes: 14
['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']


In [29]:
# model building
model.classifier = nn.Linear(
    in_features=1024,
    out_features=num_classes
)

print(model.classifier)

Linear(in_features=1024, out_features=14, bias=True)


In [30]:
model.eval()

dummy_input = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    output = model(dummy_input)

print("Output shape:", output.shape)

Output shape: torch.Size([1, 14])


In [31]:
#Loss function and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)
print("Loss function:", criterion)
print("Optimizer:", optimizer)



Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [32]:
#Freezing the features extractor
for param in model.features.parameters():
    param.requires_grad = False

print("Feature extractor frozen.")

print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

Feature extractor frozen.
Trainable parameters: 14350


In [33]:

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

classifier.weight
classifier.bias


In [34]:
#loading training validating and testing dataset
SPLIT_DIR = PROJECT_DIR / "processed"

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)
print("Validation data:", val_df.shape)

Training data: (4513, 26)
Testing data: (554, 26)
Validation data: (539, 26)


In [35]:
# image transformers
from torchvision import transforms
IMG_SIZE = 224

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=7),
    transforms.ColorJitter(brightness=0.10, contrast=0.10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])

print("Image transformations created successfully.")

Image transformations created successfully.


In [36]:
# dataset classe
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
import torch

class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, image_dir, classes, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.classes = classes
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["Image Index"]
        image = Image.open(image_path).convert("RGB")

        labels = row[self.classes].to_numpy(dtype=np.float32)
        labels = torch.tensor(labels, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, labels

In [37]:
import pandas as pd
import numpy as np
import json
import torch

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset


PROJECT_DIR = Path(r"D:\Pradeep Pandey\Nexvia Technology  Intership\Project 1")
IMAGE_DIR = PROJECT_DIR / "data" / "sample" / "images"
SPLIT_DIR = PROJECT_DIR / "processed"


train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv")


with open(SPLIT_DIR / "disease_classes.json", "r") as file:
    disease_classes = json.load(file)


class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, image_dir, classes, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.classes = classes
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = self.image_dir / row["Image Index"]
        image = Image.open(image_path).convert("RGB")

        labels = row[self.classes].to_numpy(dtype=np.float32)
        labels = torch.tensor(labels, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, labels


train_dataset = ChestXrayDataset(
    train_df, IMAGE_DIR, disease_classes, train_transform
)

val_dataset = ChestXrayDataset(
    val_df, IMAGE_DIR, disease_classes, eval_transform
)

test_dataset = ChestXrayDataset(
    test_df, IMAGE_DIR, disease_classes, eval_transform
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 4513
Validation samples: 539
Testing samples: 554


In [38]:
#Training and validation 
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

Training batches: 283
Validation batches: 34
Testing batches: 35


In [39]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

Images shape: torch.Size([16, 3, 224, 224])
Labels shape: torch.Size([16, 14])
Image data type: torch.float32
Label data type: torch.float32


In [40]:
print("Model:", type(model).__name__)
print("Loss:", criterion)
print("Optimizer:", type(optimizer).__name__)
print("Device:", device)

Model: DenseNet
Loss: BCEWithLogitsLoss()
Optimizer: Adam
Device: cpu


In [41]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()
    running_loss = 0.0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)

    return epoch_loss

In [42]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()
    running_loss = 0.0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

    epoch_loss = running_loss / len(loader)

    return epoch_loss

In [43]:
#traning loop
EPOCHS = 3

print("Number of epochs:", EPOCHS)

Number of epochs: 3


In [44]:
train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss = validate_one_epoch(
        model, val_loader, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Training Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

Epoch [1/3] Training Loss: 0.2296 | Validation Loss: 0.1685
Epoch [2/3] Training Loss: 0.1829 | Validation Loss: 0.1643
Epoch [3/3] Training Loss: 0.1808 | Validation Loss: 0.1639


In [68]:
# checkpoint saving
MODEL_DIR = PROJECT_DIR / "outputs" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "disease_classes": disease_classes,
    "architecture": "DenseNet121",
    "num_classes": num_classes,
    "image_size": IMG_SIZE
}

torch.save(
    checkpoint,
    MODEL_DIR / "densenet121_chestxray.pth"
)

print("Model checkpoint saved successfully!")

Model checkpoint saved successfully!


In [45]:
print("Model:", type(model).__name__)
print("Loss:", criterion)
print("Optimizer:", type(optimizer).__name__)
print("Device:", device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Model: DenseNet
Loss: BCEWithLogitsLoss()
Optimizer: Adam
Device: cpu
Trainable parameters: 14350


In [46]:
print("Checkpoint exists:", CHECKPOINT_PATH.exists())
print("Checkpoint size:", round(CHECKPOINT_PATH.stat().st_size / (1024**2), 2), "MB")


Checkpoint exists: True
Checkpoint size: 27.16 MB


In [55]:
# finetunning block
checkpoint = torch.load(MODEL_PATH, map_location=device)

model = models.densenet121(weights=None)
model.classifier = nn.Linear(1024, len(disease_classes))

model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)

print("Original trained model loaded successfully!")

for param in model.features.parameters():
    param.requires_grad = False


for param in model.features.denseblock4.parameters():
    param.requires_grad = True


for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Trainable parameters:", trainable_params)
optimizer = torch.optim.Adam([
    {
        "params": model.features.denseblock4.parameters(),
        "lr": 1e-5
    },
    {
        "params": model.classifier.parameters(),
        "lr": 1e-4
    }
])

criterion = nn.BCEWithLogitsLoss()

print("Optimizer:", optimizer)
print("Loss function: BCEWithLogitsLoss")


Original trained model loaded successfully!
Trainable parameters: 2172430
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 1e-05
    maximize: False
    weight_decay: 0

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)
Loss function: BCEWithLogitsLoss


In [56]:
EPOCHS = 3

train_losses_ft = []
val_losses_ft = []

print("Fine-tuning epochs:", EPOCHS)

Fine-tuning epochs: 3


In [57]:
for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss = validate_one_epoch(
        model, val_loader, criterion, device
    )

    train_losses_ft.append(train_loss)
    val_losses_ft.append(val_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Training Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

Epoch [1/3] Training Loss: 0.1845 | Validation Loss: 0.1623
Epoch [2/3] Training Loss: 0.1750 | Validation Loss: 0.1588
Epoch [3/3] Training Loss: 0.1718 | Validation Loss: 0.1595


In [59]:
FINETUNED_PATH = PROJECT_DIR / "outputs" / "models" / "densenet121_finetuned.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "disease_classes": disease_classes,
    "architecture": "DenseNet121",
    "num_classes": len(disease_classes),
    "image_size": 224
}, FINETUNED_PATH)

print("Fine-tuned checkpoint saved:", FINETUNED_PATH.exists())

Fine-tuned checkpoint saved: True
